# 35 · TiTiler: requests, tiles and processing boundaries

**Time:** 40 minutes · **Audience:** undergraduate / introductory graduate GIS and Earth science.

**By the end:** construct a current TiTiler request; diagnose a recorded service failure; distinguish numeric pixels from rendered imagery.

[Run in JupyterLite](https://jltobias.github.io/JupyterLite-zarr-sql-views/lite/lab/index.html?path=35_eopf_titiler.ipynb) · [Earth lab](https://jltobias.github.io/JupyterLite-zarr-sql-views/lab/) · [Sources and licenses](https://jltobias.github.io/JupyterLite-zarr-sql-views/book/credits.html)

Run cells from top to bottom with **Shift+Enter**. The first kernel start downloads Python WebAssembly packages.
The bundled exercises need no data-service account. Save a copy with **File → Save and Export Notebook As** before clearing browser storage.

![Four-stage workflow: STAC discovery, Zarr reading, SQL selection, linked views](assets/pipeline.svg)

In [1]:
from course import *
import numpy as np
import matplotlib.pyplot as plt
print('Ready: Python + NumPy + local teaching data')

Ready: Python + NumPy + local teaching data


**Reproducibility record.** Project: Earth in the Browser; author: course contributors;
email/organization: not supplied. Created/updated: 2026-11-21. Input: `data/eopf/`;
outputs: the figures and objects below (exports are named explicitly). Python, NumPy and
Matplotlib; SQLite where used. Initial EOPF teaching release. The acquisition is a
convenience example, not a representative sample. Missing values remain missing.

![EOPF workflow from discovery to direct maps, quality-aware analysis and server processing](assets/eopf-workflow.svg)
## 1. Build the request from the actual item

The [TiTiler guide](https://explorer.eopf.copernicus.eu/software-services/titiler/)
illustrates server-rendered RGB, indices and cropping. APIs evolve: older tutorial
band-path syntax differs from the current service's STAC asset syntax. This lesson
uses `reflectance|bands=b04,b03,b02`, checked against the service's `/raster/api`.
URL-encode the asset selection rather than manually concatenating pipe/comma escapes.

In [2]:
from eopf import load_eopf, tilejson_url, live_json, DATA
d=load_eopf(); url=tilejson_url(d['item'])
print('TileJSON request:',url)
record=json.loads((DATA/'service-provenance.json').read_text())
print('RECORDED HTTP status:',record['status'])
print('RECORDED response:',record['response'])
print('Recorded request date:',record['retrieved'])

TileJSON request: https://api.explorer.eopf.copernicus.eu/raster/collections/sentinel-2-l2a/items/S2C_MSIL2A_20260928T095031_N0513_R079_T33TVF_20260928T130916/WebMercatorQuad/tilejson.json?assets=reflectance%7Cbands%3Db04%2Cb03%2Cb02&rescale=0%2C0.4&color_formula=gamma+rgb+1.5&tile_format=png&tilesize=256
RECORDED HTTP status: 400
RECORDED response: {'detail': 'https://s3.explorer.eopf.copernicus.eu/esa-zarr-sentinel-explorer-fra/tests-output/sentinel-2-l2a/S2C_MSIL2A_20260928T095031_N0513_R079_T33TVF_20260928T130916.zarr/conditions/mask/l2a_classification/r20m/scl does not appear to be a valid GeoZarr group.'}
Recorded request date: 2026-11-21


## 2. Run one optional metadata request

TileJSON describes a tile service; constructing a URL does not prove imagery was
rendered. On 2026-11-21 this request returned **HTTP 400**: the service attempted to
open the item's SCL array as a GeoZarr group. The successful STAC and direct-array
reads in earlier lessons establish that this is not proof of missing Sentinel data.
No TiTiler preview was obtained. The next cell retries only when you choose `LIVE=True`;
it does not silently substitute another scene or a fabricated server result.

In [3]:
LIVE=False
if LIVE:
    try:
        current=await live_json(url)
        print('LIVE response:',current['tiles'][0])
    except Exception as exc:
        print('LIVE REQUEST FAILED:',exc,'The recorded response remains available.')
else:
    print('RECORDED MODE: no request sent; the saved HTTP 400 is the observed result.')

RECORDED MODE: no request sent; the saved HTTP 400 is the observed result.


## 3. Do not analyze display colors as reflectance

When a tile request succeeds, its PNG contains display colors. Its pixel grid can differ
from the UTM extract; it is not an independent validation dataset. Reprojection,
resampling, color enhancement and quantization can all change screen colors.
Use numeric GeoZarr arrays and documented masks for scientific calculations.

For an external map, open the current TiTiler Swagger UI from the guide, inspect
`assets`, `expression`, `rescale` and `reproject`, then request one small tile.
Avoid submitting an unbounded global crop. The Python kernel and map may run in a
browser while the numerical work still happens on the TiTiler server.

In [4]:
from urllib.parse import urlsplit, parse_qs
request=parse_qs(urlsplit(url).query)
assert request['assets']==['reflectance|bands=b04,b03,b02']
assert request['rescale']==['0,0.4']
print('Display request:',request)
print('Numeric extract red mean:',float(np.mean(d['b04'])))
print('No server-rendered image is used in this lesson.')

Display request: {'assets': ['reflectance|bands=b04,b03,b02'], 'rescale': ['0,0.4'], 'color_formula': ['gamma rgb 1.5'], 'tile_format': ['png'], 'tilesize': ['256']}
Numeric extract red mean: 0.11895119398832321
No server-rendered image is used in this lesson.


## Try it yourself

Change the RGB stretch in a request and explain why the vegetation did not physically change. Record processing location, acquisition time and request time separately.

## Check your reasoning

Rendering time and acquisition time are different. Brightness adjustments are display choices; underlying reflectance and quality evidence must support physical interpretations.

## Evidence journal

Write four sentences: **what I observed; how I calculated it; what this cannot establish; what evidence I need next**.
For any figure you reuse, retain its units, date or scenario, data origin, transformations, and attribution.

## Sources and credit

- [EOPF Sentinel Zarr Explorer](https://explorer.eopf.copernicus.eu/), funded by ESA through the Copernicus Space Component programme; EOX, Development Seed and project contributors.
- [Copernicus Sentinel data legal notice](https://sentinels.copernicus.eu/documents/247904/690755/Sentinel_Data_Legal_Notice): contains modified Copernicus Sentinel data (2026). EOPF processing; course subset and repackaging.
- [Recorded source objects, checksums and transformations](https://jltobias.github.io/JupyterLite-zarr-sql-views/data/eopf/provenance.json). One 128 × 128 window of the upstream 120 m averaged overview, acquired 2026-09-28.
- [TiTiler-EOPF source](https://github.com/EOPF-Explorer/titiler-eopf), Apache-2.0; Development Seed, EOPF Explorer and contributors.
- [Current raster OpenAPI](https://api.explorer.eopf.copernicus.eu/raster/api).

Teaching adaptation of [dzole0311/zarr-sql-views](https://github.com/dzole0311/zarr-sql-views), ISC.
Original teaching code and prose: JupyterLite-zarr-sql-views contributors, ISC. Data keep their separate licenses.

In [5]:
print('LESSON COMPLETE: 35_eopf_titiler')

LESSON COMPLETE: 35_eopf_titiler
